# ETL — Histórico de precios de combustibles (CNE)

Este notebook procesa el archivo `precios_comb_liquidos_en_el_pais-2026-08-03.xlsx`,
descargado de [cne.cl/estadisticas/hidrocarburo](https://www.cne.cl/estadisticas/hidrocarburo/),
para integrarlo como serie histórica de referencia en el dashboard de `combustibles-cl`.

**Fuente:** Precio Mensual Regional de Combustibles Líquidos, CNE.
**Cobertura:** Mensual, desde enero 1994 hasta la fecha de publicación del archivo.
**Nivel de agregación:** Precio promedio por región/ciudad (no por estación individual —
a diferencia de los datos capturados en tiempo real vía el pipeline de n8n).

## Pasos del ETL

1. Cargar cada una de las 5 hojas del Excel (Gasolina 93/95/97, Diésel, Kerosene),
   saltando las filas de metadata/encabezado propias del formato original.
2. Limpiar cada hoja: eliminar columna vacía, convertir fecha a tipo datetime,
   reemplazar códigos `ND`/`NE` por nulos, convertir precios a numérico.
3. Reestructurar cada hoja de formato ancho (una columna por región) a formato
   largo (una fila por combinación fecha-región), usando `pandas.melt`.
4. Unir las 5 hojas en un único DataFrame, agregando una columna `tipo_combustible`.
5. Guardar el resultado en `data/processed/historico_precios_regional.csv`.

In [1]:
import pandas as pd

archivo = '../data/raw/precios_comb_liquidos_en_el_pais-2026-08-03.xlsx'
df_93 = pd.read_excel(archivo, sheet_name='Gasolina 93 sp', header=9)
df_93.shape

(393, 18)

## Paso 1: Limpieza de la hoja "Gasolina 93 sp"

Se elimina la columna vacía (`Unnamed: 0`), se convierte la fecha a tipo
datetime, y se reemplazan los códigos `ND` (No Disponible) y `NE` (No Existe)
por valores nulos, permitiendo convertir las columnas de precio a numérico.

In [2]:
df_93 = df_93.drop(columns=['Unnamed: 0'])
df_93['Fecha'] = pd.to_datetime(df_93['Fecha'])

columnas_region = df_93.columns.drop('Fecha')
df_93[columnas_region] = df_93[columnas_region].replace(['ND', 'NE'], pd.NA)
df_93[columnas_region] = df_93[columnas_region].apply(pd.to_numeric, errors='coerce')

df_93.dtypes

Fecha             datetime64[us]
 METROPOLITANA           float64
ARICA                    float64
IQUIQUE                  float64
ANTOFAGASTA              float64
 COPIAPÓ                 float64
LA SERENA                float64
VALPARAÍSO               float64
RANCAGUA                 float64
TALCA                    float64
CHILLÁN                  float64
 CONCEPCIÓN              float64
TEMUCO                   float64
VALDIVIA                 float64
 PUERTO MONTT            float64
 COYHAIQUE               float64
 PUNTA ARENAS            float64
dtype: object

In [3]:
df_93.columns = df_93.columns.str.strip()
df_93.dtypes

Fecha            datetime64[us]
METROPOLITANA           float64
ARICA                   float64
IQUIQUE                 float64
ANTOFAGASTA             float64
COPIAPÓ                 float64
LA SERENA               float64
VALPARAÍSO              float64
RANCAGUA                float64
TALCA                   float64
CHILLÁN                 float64
CONCEPCIÓN              float64
TEMUCO                  float64
VALDIVIA                float64
PUERTO MONTT            float64
COYHAIQUE               float64
PUNTA ARENAS            float64
dtype: object

## Paso 2: Reestructurar de formato ancho a largo

Se transforma la tabla de "una columna por región" a "una fila por cada
combinación fecha-región", usando `pandas.melt`. Esto facilita el filtrado
y graficado posterior (por ejemplo, comparar la evolución de una región
específica en el tiempo).

In [4]:
df_93_largo = df_93.melt(id_vars='Fecha', var_name='region', value_name='precio')
df_93_largo['tipo_combustible'] = '93'
df_93_largo.shape

(6288, 4)

## Paso 3: Función reutilizable para procesar cada hoja

Se encapsula el proceso de limpieza (eliminar columna vacía, convertir fecha,
limpiar ND/NE, convertir a numérico, reestructurar a formato largo) en una
función, para aplicarlo de forma consistente a las 5 hojas del archivo.

**Hallazgo durante la exploración:** no todas las hojas tienen el header en
la misma fila del Excel — "Gasolina 93 sp" usa `header=9`, mientras que las
otras 4 hojas ("Gasolina 95 sp", "Gasolina 97 sp", "Petróleo Diesel",
"Kerosene Doméstico") usan `header=8`. Esta inconsistencia es propia del
archivo original de la CNE, no de un error en el procesamiento. Por esto,
la función recibe el número de fila de header como parámetro.

In [9]:
def procesar_hoja(nombre_hoja, codigo_combustible, header_fila):
    df = pd.read_excel(archivo, sheet_name=nombre_hoja, header=header_fila)
    df = df.drop(columns=['Unnamed: 0'])
    df.columns = df.columns.str.strip()
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    
    columnas_region = df.columns.drop('Fecha')
    df[columnas_region] = df[columnas_region].replace(['ND', 'NE'], pd.NA)
    df[columnas_region] = df[columnas_region].apply(pd.to_numeric, errors='coerce')
    
    df_largo = df.melt(id_vars='Fecha', var_name='region', value_name='precio')
    df_largo['tipo_combustible'] = codigo_combustible
    
    return df_largo

In [12]:
hojas = {
    'Gasolina 93 sp': ('93', 9),
    'Gasolina 95 sp': ('95', 8),
    'Gasolina 97 sp': ('97', 8),
    'Petróleo Diesel': ('DI', 8),
    'Kerosene Doméstico': ('KE', 8),
}

lista_dfs = [procesar_hoja(nombre, codigo, header_fila) for nombre, (codigo, header_fila) in hojas.items()]
df_final = pd.concat(lista_dfs, ignore_index=True)
df_final.shape

(31440, 4)

In [13]:
df_test = pd.read_excel(archivo, sheet_name='Gasolina 95 sp', header=8)
df_test.columns.tolist()[:3]

['Unnamed: 0', 'Fecha', ' METROPOLITANA']

In [14]:
df_final.head()

,Fecha,region,precio,tipo_combustible
0,1994-01-01,METROPOLITANA,166.5,93
1,1994-02-01,METROPOLITANA,166.3,93
2,1994-03-01,METROPOLITANA,166.3,93
3,1994-04-01,METROPOLITANA,170.7,93
4,1994-05-01,METROPOLITANA,170.1,93


In [15]:
df_final['tipo_combustible'].value_counts()

tipo_combustible
93    6288
95    6288
97    6288
DI    6288
KE    6288
Name: count, dtype: int64